In [3]:
pip install pandas openpyxl

Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd
import numpy as np

# === 1. Загрузка данных ===
file_path = 'Кейс 27.xlsx'
print("Чтение данных из Excel...")

sales_fact = pd.read_excel(file_path, sheet_name='SalesFact')
product_dim = pd.read_excel(file_path, sheet_name='Product')
geo_dim = pd.read_excel(file_path, sheet_name='Geo')
manufacturer_dim = pd.read_excel(file_path, sheet_name='Manufacturer')

# Фиксируем базовое количество строк и сумму выручки для последующей валидации
base_rows = len(sales_fact)
base_revenue = sales_fact['Revenue'].sum()

# === 2. BI-Валидация ключей (Защита от "раздувания" строк) ===
print("Проверка целостности справочников...")
if not product_dim['ProductID'].is_unique:
    raise ValueError("❌ ОШИБКА: В таблице Product найдены дубликаты ProductID! JOIN задвоит факты.")
if not geo_dim['Zip'].is_unique:
    raise ValueError("❌ ОШИБКА: В таблице Geo найдены дубликаты Zip! JOIN задвоит факты.")

# === 3. Объединение (Star Schema -> Flat Table) ===
print("Объединение таблиц (LEFT JOIN)...")

# Шаг 3.1: Факт + Товары
df_flat = pd.merge(
    sales_fact, 
    product_dim, 
    on='ProductID', 
    how='left', 
    suffixes=('', '_prod')
)

# Шаг 3.2: Результат + География
df_flat = pd.merge(
    df_flat, 
    geo_dim, 
    on='Zip', 
    how='left', 
    suffixes=('', '_geo')
)

# Шаг 3.3: Обогащение данными Производителя (если нужно)
# В Product уже есть ManufacturerID, поэтому джойним только уникальные поля из Manufacturer
df_flat = pd.merge(
    df_flat, 
    manufacturer_dim[['ManufacturerID', 'MfgisVanArsdel']], 
    on='ManufacturerID', 
    how='left'
)

# === 4. Обработка пропусков (BI-логика) ===
print("Обработка пропусков и форматирование...")

# Категориальные справочники заполняем маркером "Unknown", чтобы они не терялись в Pivot Tables
cat_cols = ['City', 'State', 'Region', 'District', 'Category', 'Segment', 'Manufacturer']
for col in cat_cols:
    if col in df_flat.columns:
        df_flat[col] = df_flat[col].fillna('Unknown / Not Found')

# Приведение дат к единому формату datetime (защита от проблем с форматами ММ/ДД/ГГГГ)
df_flat['Date'] = pd.to_datetime(df_flat['Date'], errors='coerce')

# === 5. Финальная валидация и сохранение ===
print("Финальная проверка потерь...")
final_rows = len(df_flat)
final_revenue = df_flat['Revenue'].sum()

if base_rows != final_rows:
    print(f"⚠️ ВНИМАНИЕ: Количество строк изменилось! Было: {base_rows}, Стало: {final_rows}")
if abs(base_revenue - final_revenue) > 0.01:
    print(f"⚠️ ВНИМАНИЕ: Сумма выручки изменилась! Было: {base_revenue}, Стало: {final_revenue}")

# Сохраняем итоговую сводную таблицу
output_file = 'VanArsdel.xlsx'
df_flat.to_excel(output_file, index=False)
print(f"✅ Успешно! Итоговая таблица сохранена в {output_file}")

Чтение данных из Excel...
Проверка целостности справочников...
Объединение таблиц (LEFT JOIN)...
Обработка пропусков и форматирование...
Финальная проверка потерь...
✅ Успешно! Итоговая таблица сохранена в VanArsdel.xlsx


In [5]:
# Сохраняем в CSV
df_flat.to_csv('VanArsdel.csv', index=False, encoding='utf-8-sig')